# Hangul Stroke Extraction — Training (BiFPN decoder)

코드는 GitHub 레포(`stroke_model` 패키지)에서 가져옵니다. 이 노트북은 실행만 담당합니다.
코드를 고칠 일이 있으면 이 노트북이 아니라 로컬(VS Code)에서 레포를 고치고 push한 뒤,
아래 "레포 받기" 셀만 다시 실행(git pull)하면 됩니다.

## 1. 레포 받기

In [ ]:
# GitHub에 게시된 HSQM 코드 받기
REPO_URL = "https://github.com/yechan25/hsqm.git"
REPO_DIR = "hsqm"

import os
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}
else:
    %cd {REPO_DIR}
    !git pull
    %cd ..

%cd {REPO_DIR}
!pip install -q -r requirements.txt


## 2. Google Drive 마운트
데이터셋/체크포인트는 코드와 분리해서 Drive에 저장합니다 (git에는 안 올림).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 3. 설정 로드

In [ ]:
from stroke_model.utils import load_config

config = load_config("configs/base.yaml")

# 필요하면 여기서 특정 값만 override 가능 (원본 yaml은 안 건드림)
# config["epochs"] = 50
# config["batch_size"] = 4

for k in ["dataset_root", "train_csv_path", "test_dataset_root", "test_csv_path", "out_dir"]:
    print(k, "=", config[k])


## 4. 데이터 입력 확인 (본격 학습 전에 한 번 실행 권장)

In [ ]:
from stroke_model.data import StrokeDataset

_preview_ds = StrokeDataset(config["train_csv_path"], config["dataset_root"], train=False, config=config)
_sample = _preview_ds[0]
print("char_id:", _sample["char_id"])
print("I_prep shape:", _sample["I_prep"].shape)
print("target_strokes shape:", _sample["target_strokes"].shape)

import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(6, 3))
ax[0].imshow(_sample["I_prep"][0], cmap="gray"); ax[0].set_title("I_prep")
ax[1].imshow(_sample["I_g_vis"][0], cmap="gray"); ax[1].set_title("I_g")
for a in ax: a.axis("off")
plt.show()


## 5. 모델 shape 점검 (학습 전 빠른 sanity check)
Swin/BiFPN 관련 shape 문제가 있으면 150 epoch 다 돌기 전에 여기서 바로 걸러집니다.

In [ ]:
import torch
from stroke_model.model import SwinWarpHintStrokeModel
from stroke_model.utils import get_device

device = get_device()
_model = SwinWarpHintStrokeModel(config).to(device)

B, K, S = 2, config["max_strokes"], config["image_size"]
_dummy_prep = torch.rand(B, 1, S, S, device=device)
_dummy_g = torch.rand(B, 1, S, S, device=device)
_dummy_strokes = torch.rand(B, K, S, S, device=device)

with torch.no_grad():
    _out = _model(_dummy_prep, _dummy_g, _dummy_strokes)

print("logits:", _out["logits"].shape)  # 기대: (B, K, S, S)
print("flow  :", _out["flow"].shape)    # 기대: (B, 2, S, S)
print("H     :", _out["H"].shape)       # 기대: (B, K, S, S)

del _model, _dummy_prep, _dummy_g, _dummy_strokes, _out
torch.cuda.empty_cache() if device.type == "cuda" else None


## 6. 학습 실행

In [ ]:
from stroke_model.train import run_training

model, history = run_training(config)


## 7. 학습 결과 확인
`config["out_dir"]`에 `checkpoints/best.pt`, `checkpoints/last.pt`, `history.json`이 저장됩니다.

In [ ]:
import json
from pathlib import Path

history_path = Path(config["out_dir"]) / "history.json"
with open(history_path) as f:
    hist = json.load(f)

epochs = [h["epoch"] for h in hist]
train_loss = [h["train"]["loss"] for h in hist]
train_dice = [h["train"]["soft_dice"] for h in hist]
test_epochs = [h["epoch"] for h in hist if h["test"] is not None]
test_dice = [h["test"]["soft_dice"] for h in hist if h["test"] is not None]

import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].plot(epochs, train_loss); ax[0].set_title("train loss")
ax[1].plot(epochs, train_dice, label="train"); ax[1].plot(test_epochs, test_dice, label="test")
ax[1].set_title("soft dice"); ax[1].legend()
plt.show()


## 8. 체크포인트 불러와서 실제 입력→출력 확인
숫자(loss/dice)만 보지 말고, 실제로 모델이 뭘 뽑아내는지 눈으로 확인합니다.

In [ ]:
from stroke_model.model import SwinWarpHintStrokeModel
from stroke_model.utils import safe_torch_load

CKPT_NAME = "best.pt"  # 다른 체크포인트 보고 싶으면 "last.pt" 또는 "epoch_0100.pt" 등으로 변경

ckpt_path = f"{config['out_dir']}/checkpoints/{CKPT_NAME}"
ckpt = safe_torch_load(ckpt_path, map_location=device)

eval_model = SwinWarpHintStrokeModel(ckpt.get("config", config)).to(device)
eval_model.load_state_dict(ckpt["model"])
eval_model.eval()

print("불러온 체크포인트:", ckpt_path)
print("epoch:", ckpt.get("epoch"), "| best_score(soft_dice):", ckpt.get("best_score"))


In [ ]:
from stroke_model.data import StrokeDataset

_test_ds_viz = StrokeDataset(config["test_csv_path"], config["test_dataset_root"], train=False, config=config)
print("test set 크기:", len(_test_ds_viz))


In [ ]:
import torch
import matplotlib.pyplot as plt


@torch.no_grad()
def visualize_prediction(idx: int, ds=_test_ds_viz, model=eval_model):
    item = ds[idx]
    I_prep = item["I_prep"][None].to(device)
    I_g = item["I_g"][None].to(device)
    I_G_strokes = item["I_G_strokes"][None].to(device)
    target = item["target_strokes"]  # K,H,W (CPU)

    out = model(I_prep, I_g, I_G_strokes)
    soft = torch.sigmoid(out["logits"])[0].cpu()  # K,H,W
    H = out["H"][0].cpu()

    # target에 실제로 획이 있는 채널만 골라서 보여줌
    active = [k for k in range(soft.shape[0]) if target[k].sum() > 1]
    if not active:
        active = list(range(min(5, soft.shape[0])))
    n = len(active)

    fig, axes = plt.subplots(3, n + 2, figsize=(2.6 * (n + 2), 7.5))

    axes[0, 0].imshow(item["I_prep"][0], cmap="gray"); axes[0, 0].set_title("I_prep (입력)")
    axes[1, 0].imshow(item["I_g_vis"][0], cmap="gray"); axes[1, 0].set_title("I_g (기준)")

    soft_union = soft[active].max(0).values
    target_union = target[active].max(0).values
    axes[0, 1].imshow(soft_union, cmap="gray", vmin=0, vmax=1); axes[0, 1].set_title("예측 획 합")
    axes[1, 1].imshow(target_union, cmap="gray", vmin=0, vmax=1); axes[1, 1].set_title("정답 획 합")

    for j, k in enumerate(active):
        axes[0, j + 2].imshow(soft[k], cmap="gray", vmin=0, vmax=1)
        axes[0, j + 2].set_title(f"예측 #{k}")
        axes[1, j + 2].imshow(target[k], cmap="gray", vmin=0, vmax=1)
        axes[1, j + 2].set_title(f"정답 #{k}")
        axes[2, j + 2].imshow(H[k], cmap="gray", vmin=0, vmax=1)
        axes[2, j + 2].set_title(f"warp hint #{k}")

    for ax in axes.flat:
        ax.axis("off")
    plt.suptitle(f"idx={idx}  char={item['char_id']}")
    plt.tight_layout()
    plt.show()


# 여러 샘플 한 번에 확인 (인덱스는 원하는 대로 바꿔도 됨)
for i in [0, 1, 2, 3, 4]:
    visualize_prediction(i)


## 9. 교차점 공유를 허용하는 graph 후처리 비교
위 8절에서 체크포인트를 불러온 뒤 실행합니다. clone한 저장소에도 새 `graph_postprocess.py`가 있어야 합니다.
정답은 평가에만 사용하며, 추론에는 기준 획 채널과 모델의 warp된 H만 전달합니다.
기본값은 검증 전 시작값입니다. 소수 샘플 시각화는 전체 데이터셋 성능 검증을 대체하지 않습니다.

- raw / 기존 후처리 / graph(H 없이) / graph(H 포함) / 정답을 같은 입력 잉크 영역에서 비교합니다.
- Dice/IoU는 실제 정답 획만 평균내며, 입력 잉크 밖 정답 비율도 별도로 출력합니다.
- 교차점 중복은 허용합니다. 입력 coverage=100%를 강제하지 않습니다.


In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from stroke_model.graph_postprocess import postprocess_batch_predictions_graph
from stroke_model.postprocess import (
    postprocess_batch_predictions_seeded_partition, make_input_ink_binary,
)
from stroke_model.data import StrokeDataset, set_ignore_dirnames

_graph_config = dict(ckpt.get("config", config))
set_ignore_dirnames(_graph_config.get("ignore_dirnames", []))
_graph_ds = StrokeDataset(config["test_csv_path"], config["test_dataset_root"],
                          train=False, config=_graph_config)
GRAPH_THRESHOLD = float(_graph_config.get("hard_threshold", 0.25))
GRAPH_SMOOTHNESS = 1.0
GRAPH_HINT_WEIGHT = 0.25
GRAPH_EDGE_SCALE = 0.25


def _graph_metrics(mask, target, ink):
    gt = target & ink[None]
    valid = gt.sum((1, 2)) > 0
    intersection = (mask & gt).sum((1, 2))
    denominator = mask.sum((1, 2)) + gt.sum((1, 2))
    union = (mask | gt).sum((1, 2))
    return {
        "active_dice": float((2 * intersection[valid] / denominator[valid]).mean()) if valid.any() else np.nan,
        "active_iou": float((intersection[valid] / union[valid]).mean()) if valid.any() else np.nan,
        "missed_strokes": int((valid & (intersection == 0)).sum()),
        "inactive_channel_pixels": int(mask[~(target.sum((1, 2)) > 0)].sum()),
        "ink_coverage": float((mask.any(0) & ink).sum() / max(1, ink.sum())),
        "shared_pixels": int((mask.sum(0) > 1).sum()),
    }


@torch.no_grad()
def compare_graph_postprocessing(idx):
    item = _graph_ds[idx]
    prep = item["I_prep"][None].to(device)
    ref_img = item["I_g"][None].to(device)
    refs = item["I_G_strokes"][None].to(device)
    out = eval_model(prep, ref_img, refs)
    soft = torch.sigmoid(out["logits"])
    ink = make_input_ink_binary(item["I_prep"][0].numpy())
    active = (refs[0].sum((1, 2)) > 0).cpu().numpy()
    target = item["target_strokes"].numpy() > GRAPH_THRESHOLD
    raw = (soft[0].cpu().numpy() > GRAPH_THRESHOLD) & ink[None] & active[:, None, None]
    masks = {"raw on ink": raw}
    times = {"raw on ink": 0.0}
    start = time.perf_counter()
    legacy = postprocess_batch_predictions_seeded_partition(soft, prep, target_hints=refs)
    masks["legacy exclusive"] = legacy[0].cpu().numpy() > 0.5
    times["legacy exclusive"] = time.perf_counter() - start
    for name, weight in [("graph no hint", 0.0), ("graph + warped H", GRAPH_HINT_WEIGHT)]:
        start = time.perf_counter()
        result = postprocess_batch_predictions_graph(
            soft, prep, reference_strokes=refs, warped_hints=out["H"],
            mode="overlap", threshold=GRAPH_THRESHOLD, smoothness=GRAPH_SMOOTHNESS,
            hint_weight=weight, edge_scale=GRAPH_EDGE_SCALE,
        )
        masks[name] = result[0].cpu().numpy() > 0.5
        times[name] = time.perf_counter() - start
    rows = [{"method": name, **_graph_metrics(mask, target, ink), "seconds": times[name]}
            for name, mask in masks.items()]
    print(f"idx={idx}, char={item['char_id']}")
    print("Target pixels outside input ink:", round(float((target & ~ink[None]).sum() / max(1, target.sum())), 4))
    print("GT channels with no pixels inside ink:", np.flatnonzero((target.sum((1,2)) > 0) & ((target & ink[None]).sum((1,2)) == 0)).tolist())
    display(pd.DataFrame(rows))
    masks["GT on ink"] = target & ink[None]
    ids = np.flatnonzero(active | (target.sum((1, 2)) > 0))
    fig, axes = plt.subplots(len(masks), len(ids) + 1, squeeze=False,
                             figsize=(2.2 * (len(ids) + 1), 2.1 * len(masks)))
    for r, (name, mask) in enumerate(masks.items()):
        axes[r, 0].imshow(mask.sum(0), cmap="viridis", vmin=0, vmax=max(2, len(ids)))
        axes[r, 0].set_title(name + " / count")
        for c, k in enumerate(ids, 1):
            axes[r, c].imshow(mask[k], cmap="gray", vmin=0, vmax=1)
            axes[r, c].set_title(f"stroke {k}")
    for ax in axes.flat:
        ax.axis("off")
    plt.tight_layout()
    plt.show()
    return rows


_graph_sample_rows = []
for idx in range(min(5, len(_graph_ds))):
    for row in compare_graph_postprocessing(idx):
        _graph_sample_rows.append({"sample": idx, **row})
